# Pauli Algebra 05: Model Builders

This notebook surveys the Pauli-backed model classes that build common spin and fermion Hamiltonians as `PauliSum` objects.


## What you will learn

1. Which ready-made Pauli model builders are available.
2. How to inspect term counts, matrix shapes, and Hermiticity on small examples.
3. How symbolic Hamiltonian methods can be used for notebook documentation.
4. Which placeholder is intentionally unsupported in the current qubit backend.


In [1]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import types


def _ensure_ipython_display_if_missing():
    try:
        import IPython.display  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    ipython_module = types.ModuleType("IPython")
    display_module = types.ModuleType("IPython.display")
    display_module.display = lambda *args, **kwargs: None
    display_module.Math = lambda *args, **kwargs: None
    ipython_module.display = display_module
    ipython_module.get_ipython = lambda: None
    ipython_module.version_info = (0, 0)
    sys.modules.setdefault("IPython", ipython_module)
    sys.modules.setdefault("IPython.display", display_module)


def _import_pauli_algebra():
    _ensure_ipython_display_if_missing()
    return importlib.import_module("quantum_simulation.pauli_algebra")


try:
    pa = _import_pauli_algebra()
except ModuleNotFoundError as exc:
    if exc.name != "quantum_simulation":
        raise
    repo_dir = Path("quantum-simulation")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/ToelUl/quantum-simulation.git"], check=True)
    target = Path("quantum_simulation")
    if not target.exists():
        shutil.copytree(repo_dir / "quantum_simulation", target)
    pa = _import_pauli_algebra()

print("Loaded quantum_simulation.pauli_algebra from", Path(pa.__file__).parent)


Loaded quantum_simulation.pauli_algebra from /mnt/d/phd_research_projects/quantum-simulation-main/quantum_simulation/pauli_algebra


In [2]:
import numpy as np

from quantum_simulation.pauli_algebra import (
    PauliHeisenbergModel,
    PauliIsingModel,
    PauliHeisenbergModel2D,
    PauliIsingModel2D,
    PauliKitaevHoneycombModel,
    PauliKitaevChain,
    PauliSSHModel,
    PauliHubbardModel1D,
    PauliTJModel2D,
    PauliBoseHubbardModel2D,
)

np.set_printoptions(precision=4, suppress=True)


## A helper for small model diagnostics

All models inherit from `PauliHamiltonian`, so they share `build`, `to_matrix`, `num_terms`, and Lie-dynamics helpers.


In [3]:
def summarize_model(name, model, *, check_hermitian=True):
    H = model.build()
    mat = model.to_matrix()
    hermitian = np.allclose(mat, mat.conj().T)
    row = {
        "name": name,
        "qubits": model.lattice_length,
        "terms": model.num_terms(),
        "matrix_shape": mat.shape,
        "hermitian": hermitian,
    }
    print(row)
    if check_hermitian:
        assert hermitian
    return H, mat


## 1D spin models

The 1D Ising and Heisenberg builders add nearest-neighbor spin terms and optional fields.


In [4]:
ising_1d = PauliIsingModel(lattice_length=4, j_coupling=1.0, h_field=0.7, pbc=False)
heisenberg_1d = PauliHeisenbergModel(
    lattice_length=3,
    jx=1.0,
    jy=0.8,
    jz=0.6,
    hz=0.2,
    pbc=False,
)

H_ising_1d, _ = summarize_model("Ising 1D", ising_1d)
H_heisenberg_1d, _ = summarize_model("Heisenberg 1D", heisenberg_1d)
print("Ising 1D Hamiltonian:", H_ising_1d)


{'name': 'Ising 1D', 'qubits': 4, 'terms': 7, 'matrix_shape': (16, 16), 'hermitian': True}
{'name': 'Heisenberg 1D', 'qubits': 3, 'terms': 9, 'matrix_shape': (8, 8), 'hermitian': True}
Ising 1D Hamiltonian: ((-0.25+0j))*XXII + ((-0.25+0j))*IXXI + ((-0.25+0j))*IIXX + ((-0.35+0j))*ZIII + ((-0.35+0j))*IZII + ((-0.35+0j))*IIZI + ((-0.35+0j))*IIIZ


## 2D rectangular spin models

For `BasePauliModel2D`, coordinates map to a 1D site index by `site = y * width + x`.


In [5]:
ising_2d = PauliIsingModel2D(width=2, height=2, j_coupling=1.0, h_field=0.4, pbc=False)
heisenberg_2d = PauliHeisenbergModel2D(
    width=2,
    height=2,
    jx=1.0,
    jy=0.5,
    jz=0.25,
    hx=0.0,
    hy=0.0,
    hz=0.2,
    pbc=False,
)

summarize_model("Ising 2D", ising_2d)
summarize_model("Heisenberg 2D", heisenberg_2d)
print("coordinate (x=1, y=1) maps to site", ising_2d._map_coord_to_index(1, 1))


{'name': 'Ising 2D', 'qubits': 4, 'terms': 8, 'matrix_shape': (16, 16), 'hermitian': True}
{'name': 'Heisenberg 2D', 'qubits': 4, 'terms': 16, 'matrix_shape': (16, 16), 'hermitian': True}
coordinate (x=1, y=1) maps to site 3


## Honeycomb and spinless fermion models

The honeycomb Kitaev model has two sublattices per unit cell. The Kitaev chain and SSH model are fermionic models built through Jordan-Wigner mappings.


In [6]:
kitaev_honeycomb = PauliKitaevHoneycombModel(u_cells=2, v_cells=1, jx=1.0, jy=0.8, jz=0.6, pbc=False)
kitaev_chain = PauliKitaevChain(lattice_length=3, chemical_potential=0.4, hopping=1.0, pairing_gap=0.3, pbc=False)
ssh = PauliSSHModel(num_cells=2, intra_cell_hopping=1.0, inter_cell_hopping=0.4, pbc=False)

summarize_model("Kitaev honeycomb", kitaev_honeycomb)
summarize_model("Kitaev chain", kitaev_chain)
summarize_model("SSH", ssh)

h_sp = ssh.build_single_particle_hamiltonian()
print("SSH single-particle Hamiltonian:")
print(h_sp)
assert h_sp.shape == (4, 4)


{'name': 'Kitaev honeycomb', 'qubits': 4, 'terms': 3, 'matrix_shape': (16, 16), 'hermitian': True}
{'name': 'Kitaev chain', 'qubits': 3, 'terms': 7, 'matrix_shape': (8, 8), 'hermitian': True}
{'name': 'SSH', 'qubits': 4, 'terms': 6, 'matrix_shape': (16, 16), 'hermitian': True}
SSH single-particle Hamiltonian:
tensor([[ 0.0000+0.j, -1.0000+0.j,  0.0000+0.j,  0.0000+0.j],
        [-1.0000+0.j,  0.0000+0.j, -0.4000+0.j,  0.0000+0.j],
        [ 0.0000+0.j, -0.4000+0.j,  0.0000+0.j, -1.0000+0.j],
        [ 0.0000+0.j,  0.0000+0.j, -1.0000+0.j,  0.0000+0.j]])


## Spinful fermion models

`PauliHubbardModel1D` and `PauliTJModel2D` use two Jordan-Wigner modes per physical site. Here the t-J example is pure exchange, which is a compact Hermitian diagnostic case.


In [7]:
hubbard = PauliHubbardModel1D(lattice_length=2, hopping=1.0, interaction=2.0, chemical_potential=0.5, pbc=False)
tj_exchange = PauliTJModel2D(width=2, height=1, hopping=0.0, exchange=0.2, pbc=False)

summarize_model("Hubbard 1D", hubbard)
summarize_model("t-J 2D exchange-only", tj_exchange)
print("Hubbard physical sites:", hubbard.physical_L, "JW modes/qubits:", hubbard.lattice_length)


{'name': 'Hubbard 1D', 'qubits': 4, 'terms': 10, 'matrix_shape': (16, 16), 'hermitian': True}
{'name': 't-J 2D exchange-only', 'qubits': 4, 'terms': 15, 'matrix_shape': (16, 16), 'hermitian': True}
Hubbard physical sites: 2 JW modes/qubits: 4


## Symbolic Hamiltonian helpers

Model classes also provide `get_symbolic_hamiltonian()` for documentation and notebooks.


In [9]:
print("Ising symbolic Hamiltonian:")
ising_1d.get_symbolic_hamiltonian()

Ising symbolic Hamiltonian:


-J*Sum(S^x[i + 1]*S^x[i], (i, 1, N - 1)) - h*Sum(S^z[i], (i, 1, N))

In [10]:
print("SSH symbolic Hamiltonian:")
ssh.get_symbolic_hamiltonian()

SSH symbolic Hamiltonian:


-v*\sum_{i}(c^\dagger_A[i]*c_B[i] + c^\dagger_B[i]*c_A[i]) - w*\sum_{i}(c^\dagger_A[i + 1]*c_B[i] + c^\dagger_B[i]*c_A[i + 1])

## Intentional unsupported placeholder

`PauliBoseHubbardModel2D` is intentionally unsupported because a general Bose-Hubbard local Hilbert space is not a qubit Pauli backend.


In [11]:
try:
    PauliBoseHubbardModel2D(width=2, height=2, hopping=1.0, interaction=1.0)
except NotImplementedError as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0] + ".")
else:
    raise AssertionError("PauliBoseHubbardModel2D should not be implemented here")


NotImplementedError
PauliBoseHubbardModel2D is not supported in the current qubit Pauli backend.
